# Lesson 04 - Conversations with `AgentSession`

**Level:** intermediate | **Demo time:** 8 minutes | **Builds on:** Lesson 02

Each call without a session is an independent conversation. Reusing an `AgentSession` lets an agent resolve references such as "it," "that order," or "the earlier option" without manually rebuilding the message list.

## Learning objectives

- Create and reuse an `AgentSession` across turns.
- Serialize session state without displaying opaque conversation identifiers.
- Understand that applications must authorize access before resuming a session.

## Mental model

An agent definition describes **how** the agent behaves. A session represents **one conversation's state**. A production application normally stores a session per authenticated user, case, or task.

> Security note: a service-side conversation ID is not authorization. Keep it in trusted storage and verify the caller before resuming it.

In [ ]:
from agent_framework import Agent, AgentSession

from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)
from maf_foundry_hackathon.support_tools import lookup_order, search_support_policy

settings = WorkshopSettings.from_env()
credential = create_credential()

conversation_agent = Agent(
    client=create_chat_client(settings, credential),
    name="conversation_support_agent",
    instructions=(
        "You are a concise support agent. Use tools for order and policy facts. "
        "Resolve follow-up references from the current conversation only."
    ),
    tools=[lookup_order, search_support_policy],
)

In [ ]:
session = conversation_agent.create_session()
print("New session created:", type(session).__name__)

In [ ]:
first_turn = await conversation_agent.run(
    "Please check order ORD-1002 and tell me its current status.",
    session=session,
)
print("Agent:", first_turn.text)

In [ ]:
second_turn = await conversation_agent.run(
    "Can I cancel it now?",
    session=session,
)
print("Agent:", second_turn.text)

The second message contains no order ID. Because both calls use the same session, the agent can connect "it" to `ORD-1002` and use the shipping policy tool.

In [ ]:
serialized_session = session.to_dict()
print("Serializable top-level fields:", sorted(serialized_session))
print("Opaque IDs and message contents are intentionally not printed.")

In [ ]:
restored_session = AgentSession.from_dict(serialized_session)
third_turn = await conversation_agent.run(
    "Summarize the status and cancellation guidance in one sentence.",
    session=restored_session,
)
print("Agent:", third_turn.text)

## Production considerations

- Associate stored sessions with an authenticated user and tenant.
- Encrypt persisted state and set retention limits.
- Do not share a mutable session across unrelated concurrent conversations.
- Compact or summarize long histories to control cost and context size.
- Start a new session when purpose, user, or authorization scope changes.

In [ ]:
credential.close()

## Recap

The first five lessons define agent behavior in Python. Lesson 05 switches ownership: it connects MAF to an agent definition already managed and versioned in Microsoft Foundry.